# Transformer-Based Sentiment Analysis — Full Corpus

**Notebook 2b of 4** — LA Reddit Community Analysis, GBA 6410 Group 7

> **Requires a GPU runtime.** Runtime → Change runtime type → T4 GPU.

Notebook 2 established the methodology on a 100,000-record sample. This notebook applies both transformer models to all 498,147 comments, so that every method in the comparison covers the same records and no scope caveat is needed when the results are reported.

**Why this notebook is separate.** Scoring the full corpus takes 40–80 minutes across two models, long enough that a session disconnect is a realistic risk. Everything here is therefore written to be resumable: each model saves after every block of 25,000 records, and re-running a cell after an interruption picks up from the last saved block rather than starting over.

**Input:** `corpus_scored_lexicon.parquet` from notebook 1
**Output:** `transformer_scores_full.parquet` — both models' labels and confidences for all 498,147 records

## Setup

In [2]:
# Mount Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install -q --upgrade transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 82.7 MB/s eta 0:00:00


## Configuration

The shared config block, with full-corpus paths and the chunk size appended.

`CHUNK = 25_000` sets the checkpoint interval. Smaller chunks mean less lost work if a session drops, at the cost of slightly more frequent writes to Drive. Twenty blocks per model proved a reasonable balance.

In [4]:
# ==================================================================
#  SHARED CONFIG — identical in every notebook in this series.
#  Edit here, then copy this whole cell to the other notebooks.
#  The fingerprint assertion below catches a stale copy.
# ==================================================================
import re, html, hashlib, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# --- Analysis constants -------------------------------------------
RANDOM_SEED    = 42
SAMPLE_N       = 100_000
POS_T, NEG_T   = 0.05, -0.05    # lexicon class thresholds
CONF_THRESHOLD = 0.50           # topic assignment confidence cut

# --- Corpus facts (verified against the preprocessing output) ------
EXPECTED_ROWS = 498_147
EXPECTED_COLS = 13

# --- Paths --------------------------------------------------------
BASE = Path("/content/drive/MyDrive/GBA6410")
BASE.mkdir(parents=True, exist_ok=True)

PATHS = {
    # inputs
    "raw":          BASE / "comments_clean_final.csv",
    "topics":       BASE / "topic_assignments.parquet",
    # notebook 01 outputs
    "sample":       BASE / "sample_100k_stage1.parquet",
    "scored":       BASE / "corpus_scored_lexicon.parquet",
    "lex_full":     BASE / "lexicon_scores_full.parquet",
    "lex_sample":   BASE / "lexicon_scores_sample.parquet",
    # notebook 02 outputs
    "transformer":  BASE / "transformer_scores.parquet",
}

# --- Drift guard --------------------------------------------------
# Any change to a constant above changes this hash. Compared against
# the literal value printed by notebook 01, so a stale copy of this
# cell fails loudly instead of silently scoring a different corpus.
CONFIG_FINGERPRINT = hashlib.md5(
    f"{RANDOM_SEED}|{SAMPLE_N}|{POS_T}|{NEG_T}|{CONF_THRESHOLD}".encode()
).hexdigest()[:8]

assert CONFIG_FINGERPRINT == "ae74dc34", (
    f"Config drift: got {CONFIG_FINGERPRINT}, expected ae74dc34. "
    "Re-copy the config cell from notebook 01."
)

np.random.seed(RANDOM_SEED)
pd.set_option("display.max_colwidth", 150)
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

print(f"Config fingerprint: {CONFIG_FINGERPRINT}")
print(f"Base: {BASE}")

# --- notebook 02b specifics ---------------------------------------
BATCH_SIZE = 64
MAX_LENGTH = 512
CHUNK      = 25_000        # records per checkpoint

MODELS = {
    "roberta": "cardiffnlp/twitter-roberta-base-sentiment-latest",
    "bert":    "nlptown/bert-base-multilingual-uncased-sentiment",
}

PATHS["roberta_full"]     = BASE / "full_roberta_scores.parquet"
PATHS["bert_full"]        = BASE / "full_bert_scores.parquet"
PATHS["transformer_full"] = BASE / "transformer_scores_full.parquet"

STAR_TO_CLASS = {1: "negative", 2: "negative", 3: "neutral",
                 4: "positive", 5: "positive"}
print("Config OK.")

Config fingerprint: ae74dc34
Base: /content/drive/MyDrive/GBA6410
Config OK.


## Preflight checks

The GPU assertion stops the notebook immediately rather than letting it run for many hours on CPU. Notebook 2's output is also required, because the sample-versus-full comparison at the end reads from it.

In [5]:
import torch
from transformers import pipeline

assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

assert PATHS["scored"].exists(), f"Missing: {PATHS['scored']} (run Notebook 01)"
assert PATHS["transformer"].exists(), f"Missing: {PATHS['transformer']} (run Notebook 02)"

_probe = BASE / ".write_test"
_probe.write_text("ok"); assert _probe.read_text() == "ok"; _probe.unlink()
print("OK: inputs present, Drive writable")

GPU: Tesla T4
OK: inputs present, Drive writable


## Normalising labels

This notebook is standalone, so it carries its own copy of the label mapping. The assertions verify the mapping before any GPU time is spent — a checkpoint returning an unexpected label format would otherwise surface partway through a long run.

In [6]:
def normalize_label(raw):
    """Map any checkpoint's output label to negative/neutral/positive."""
    s = str(raw).strip().lower()
    if "star" in s:
        return STAR_TO_CLASS[int(s.split()[0])]
    if s in {"negative", "neutral", "positive"}:
        return s
    if s.startswith("label_"):
        return ["negative", "neutral", "positive"][int(s.split("_")[1])]
    raise ValueError(f"Unrecognized label: {raw!r}")

assert normalize_label("2 stars")  == "negative"
assert normalize_label("3 stars")  == "neutral"
assert normalize_label("POSITIVE") == "positive"
assert normalize_label("LABEL_1")  == "neutral"
print("Label mapping OK")

Label mapping OK


## Scoring functions

These differ from notebook 2 in one structural respect: **the pipeline is created once and passed in**, rather than being built inside each scoring call. Notebook 2 called each scorer once, so building the model internally was harmless. Here each scorer is called roughly twenty times, once per chunk, and rebuilding the model each time would dominate the runtime.

Everything else carries over. Records are sorted by length so that each batch pads to a similar width, and results are written back through the `order` array so every score lands on its own record. The BERT scorer sums star probabilities into each class rather than mapping the single most likely star — the reasoning is documented in notebook 2, where the two conversions were compared directly.

In [7]:
def make_pipe(model_name, top_k=None):
    return pipeline("sentiment-analysis", model=model_name, device=0,
                    torch_dtype=torch.float16, top_k=top_k)

def score_with_pipe(pipe, texts, batch_size=BATCH_SIZE, desc=""):
    """3-class checkpoint (RoBERTa). Returns (labels, confs) in input order."""
    n = len(texts)
    order  = np.argsort([len(t) for t in texts])
    stexts = [texts[i] for i in order]

    raw = []
    for i in tqdm(range(0, n, batch_size), desc=desc, leave=False):
        raw.extend(pipe(stexts[i:i + batch_size],
                        truncation=True, max_length=MAX_LENGTH))

    labels, confs = [None] * n, [None] * n
    for pos, orig in enumerate(order):
        r = raw[pos][0] if isinstance(raw[pos], list) else raw[pos]
        labels[orig] = normalize_label(r["label"])
        confs[orig]  = r["score"]
    return labels, confs

def score_bert_with_pipe(pipe, texts, batch_size=BATCH_SIZE, desc=""):
    """5-star checkpoint, collapsed by SUMMING star probabilities.

    P(negative) = P(1*) + P(2*), etc. Correct because the stars are an
    ordinal scale; taking the argmax star instead lets 3* win on a
    plurality while negative mass sits split across two adjacent bins.
    """
    n = len(texts)
    order  = np.argsort([len(t) for t in texts])
    stexts = [texts[i] for i in order]

    raw = []
    for i in tqdm(range(0, n, batch_size), desc=desc, leave=False):
        raw.extend(pipe(stexts[i:i + batch_size],
                        truncation=True, max_length=MAX_LENGTH))

    labels, confs = [None] * n, [None] * n
    for pos, orig in enumerate(order):
        agg = {"negative": 0.0, "neutral": 0.0, "positive": 0.0}
        for d in raw[pos]:
            agg[STAR_TO_CLASS[int(str(d["label"]).split()[0])]] += d["score"]
        best = max(agg, key=agg.get)
        labels[orig], confs[orig] = best, agg[best]
    return labels, confs

## Resumable chunked scoring

This is what makes a long run survivable. Before starting, the function reads whatever has already been saved and skips those records. After each block it writes the accumulated results back to disk.

If a session disconnects, re-running the same cell resumes from the last completed block. At most one block of 25,000 records is lost, rather than the entire run.

The failure this guards against is not only disconnection. Requesting the full probability distribution from the five-star model returns five dictionaries per record, so a 25,000-record block holds 125,000 of them before reduction — enough that memory pressure is a genuine consideration at full corpus scale.

In [8]:
def score_chunked(ids, texts, pipe, scorer, prefix, out_path):
    """Score in chunks, checkpointing after each. Re-run the cell to resume."""
    done = pd.read_parquet(out_path) if out_path.exists() else pd.DataFrame()
    done_ids = set(done["comment_id"]) if len(done) else set()

    todo = [(c, t) for c, t in zip(ids, texts) if c not in done_ids]
    print(f"{prefix}: {len(done_ids):,} done | {len(todo):,} to go")

    for start in range(0, len(todo), CHUNK):
        part  = todo[start:start + CHUNK]
        cids  = [c for c, _ in part]
        ctext = [t for _, t in part]

        labels, confs = scorer(pipe, ctext,
                               desc=f"{prefix} {start:,}-{start + len(part):,}")

        done = pd.concat([done, pd.DataFrame({
            "comment_id": cids,
            f"{prefix}_label": labels,
            f"{prefix}_conf":  confs,
        })], ignore_index=True)
        done.to_parquet(out_path, index=False)
        print(f"  saved {len(done):,} / {len(ids):,}")

    return done

## Loading the corpus

Only the identifier and text columns are needed. The dataframe is discarded once the two lists are extracted, since holding a redundant copy alongside the model and its activations is the most likely cause of a memory failure here.

In [9]:
full = pd.read_parquet(PATHS["scored"], columns=["comment_id", "sent_text"])
assert len(full) == EXPECTED_ROWS, f"Expected {EXPECTED_ROWS:,}, got {len(full):,}"
assert not full["comment_id"].duplicated().any(), "Duplicate comment_id"

FULL_IDS  = full["comment_id"].tolist()
FULL_TEXT = full["sent_text"].tolist()
del full
print(f"Ready: {len(FULL_IDS):,} comments")

Ready: 498,147 comments


## Scoring with Twitter-RoBERTa

Roughly 20–40 minutes. The models are run in separate cells so that a failure in one does not require re-running the other.

The Hugging Face warning about unauthenticated requests in the output below is expected and harmless — it notes that anonymous downloads have lower rate limits, which does not affect a run of this size.

In [ ]:
pipe = make_pipe(MODELS["roberta"])
rob  = score_chunked(FULL_IDS, FULL_TEXT, pipe, score_with_pipe,
                     "roberta", PATHS["roberta_full"])
del pipe; torch.cuda.empty_cache()
print(rob["roberta_label"].value_counts(normalize=True).round(4))

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

  saved 450,000 / 498,147


roberta 450,000-475,000:   0%|          | 0/391 [00:00<?, ?it/s]

  saved 475,000 / 498,147


roberta 475,000-498,147:   0%|          | 0/362 [00:00<?, ?it/s]

  saved 498,147 / 498,147
roberta_label
negative    0.5212
neutral     0.3490
positive    0.1299
Name: proportion, dtype: float64


**Result.** Twitter-RoBERTa over the full corpus: 13.0% positive, 34.9% neutral, 52.1% negative.

Compared with the sample figures from notebook 2 (13.14 / 35.00 / 51.86), the largest movement is 0.26 percentage points.

## Scoring with multilingual BERT

`top_k=None` requests the full five-class distribution rather than the top rating, which is what the summed conversion requires.

In [10]:
pipe = make_pipe(MODELS["bert"], top_k=None)
bert = score_chunked(FULL_IDS, FULL_TEXT, pipe, score_bert_with_pipe,
                     "bert", PATHS["bert_full"])
del pipe; torch.cuda.empty_cache()
print(bert["bert_label"].value_counts(normalize=True).round(4))

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

  saved 450,000 / 498,147


bert 450,000-475,000:   0%|          | 0/391 [00:00<?, ?it/s]

  saved 475,000 / 498,147


bert 475,000-498,147:   0%|          | 0/362 [00:00<?, ?it/s]

  saved 498,147 / 498,147
bert_label
negative    0.6262
positive    0.2925
neutral     0.0813
Name: proportion, dtype: float64


**Result.** Multilingual BERT over the full corpus: 29.3% positive, 8.1% neutral, 62.6% negative.

Compared with the sample figures (29.22 / 8.32 / 62.45), the largest movement is 0.19 percentage points.

## Merging the two models

The merge is written to tolerate a partial run. If one model has not finished — because a session ended before its final block — the length check catches it and the notebook falls back to the completed model rather than silently joining an incomplete file and dropping records.

`validate="one_to_one"` raises if either side contains duplicate keys, rather than quietly multiplying rows.

In [11]:
# ---- Merge whatever is available ---------------------------------
# Tolerates a partial BERT run: a length check prevents an incomplete
# file from silently truncating the merged output.

rob = pd.read_parquet(PATHS["roberta_full"])
assert len(rob) == EXPECTED_ROWS, f"RoBERTa incomplete: {len(rob):,}"

if PATHS["bert_full"].exists():
    bert = pd.read_parquet(PATHS["bert_full"])
    if len(bert) == EXPECTED_ROWS:
        tf_full = rob.merge(bert, on="comment_id", how="inner", validate="one_to_one")
        print("Merged RoBERTa + BERT")
    else:
        tf_full = rob
        print(f"BERT partial ({len(bert):,}/{EXPECTED_ROWS:,}) — RoBERTa only. "
              "Re-run the BERT scoring cell to finish.")
else:
    tf_full = rob
    print("RoBERTa only (BERT not yet run)")

assert len(tf_full) == EXPECTED_ROWS
assert tf_full.notna().all().all()
tf_full.to_parquet(PATHS["transformer_full"], index=False)
print(f"Saved {len(tf_full):,} rows, {tf_full.shape[1]} cols")

Merged RoBERTa + BERT
Saved 498,147 rows, 5 cols


## Full-corpus distributions and agreement

The reporting is conditional on which models are present, so this cell runs correctly whether BERT has been scored or not.

In [12]:
has_bert = "bert_label" in tf_full.columns

cols = {"RoBERTa": tf_full["roberta_label"].value_counts(normalize=True)}
if has_bert:
    cols["BERT"] = tf_full["bert_label"].value_counts(normalize=True)

print("Full corpus label shares (%):")
print((pd.DataFrame(cols).reindex(["positive", "neutral", "negative"]).fillna(0) * 100).round(2))

if has_bert:
    print(f"\nTransformer agreement: "
          f"{(tf_full['roberta_label'] == tf_full['bert_label']).mean():.1%}")
    print(pd.crosstab(tf_full["roberta_label"], tf_full["bert_label"],
                      rownames=["RoBERTa"], colnames=["BERT"]))

Full corpus label shares (%):
          RoBERTa   BERT
positive    12.99  29.25
neutral     34.90   8.13
negative    52.12  62.62

Transformer agreement: 56.2%
BERT      negative  neutral  positive
RoBERTa                              
negative    208533    14181     36909
neutral      93281    21543     59014
positive     10110     4786     49790


**Result.** Full corpus, all 498,147 comments:

| | RoBERTa | BERT |
|---|---|---|
| positive | 12.99% | 29.25% |
| neutral | 34.90% | 8.13% |
| negative | 52.12% | 62.62% |

The two transformers agree on **56.2%** of comments — essentially identical to the 56.2% measured on the sample, and barely above the 55.9% agreement between the two lexicon methods in notebook 1. Model family does not determine agreement.

The crosstab shows where the disagreement sits:

```
BERT        negative  neutral  positive
RoBERTa
negative     208,533   14,181    36,909
neutral       93,281   21,543    59,014
positive      10,110    4,786    49,790
```

Two figures worth carrying forward. Of RoBERTa's 173,838 neutral predictions, BERT labels only 21,543 neutral — **12.4%**. And 47,019 comments, **9.4% of the corpus**, receive directly opposite pole assignments from the two models.

## Validating the sampling design

Earlier stages of the analysis ran on a 100,000-record random sample. Now that the full corpus has been scored with the same models, the two can be compared directly — turning an assumption about representativeness into a measurement.

This matters beyond tidiness: the human-coded benchmark in notebook 3 is drawn by the same random sampling approach, so evidence that the approach works supports that design too.

This cell is CPU-only and takes seconds.

In [ ]:
# ---- Sample vs full corpus: was the 100K sample representative? ----
samp = pd.read_parquet(PATHS["transformer"])
CLASSES = ["positive", "neutral", "negative"]

rows = []
for m in ["roberta", "bert"]:
    if f"{m}_label" not in tf_full.columns:
        continue
    f = tf_full[f"{m}_label"].value_counts(normalize=True).reindex(CLASSES)
    s = samp[f"{m}_label"].value_counts(normalize=True).reindex(CLASSES)
    for c in CLASSES:
        rows.append({"model": m, "class": c,
                     "sample %": s[c] * 100,
                     "full %":   f[c] * 100,
                     "diff (pp)": (s[c] - f[c]) * 100})

val = pd.DataFrame(rows).set_index(["model", "class"]).round(2)
print(val)
print(f"\nMax absolute deviation: {val['diff (pp)'].abs().max():.2f} pp")

**Expected result.** Both models should show a maximum deviation of roughly a quarter of a percentage point — 0.26 for RoBERTa and 0.19 for BERT, based on the figures reported above.

A deviation of that size is well inside the sampling error for 100,000 records drawn from 498,147, and confirms the sample was representative of the corpus on the quantity being measured.

---
## Outputs

| File | Contents |
|---|---|
| `full_roberta_scores.parquet` | Twitter-RoBERTa labels and confidences, 498,147 records (checkpoint) |
| `full_bert_scores.parquet` | Multilingual BERT, summed star collapse, 498,147 records (checkpoint) |
| `transformer_scores_full.parquet` | Both models merged on `comment_id` — the file notebook 3 consumes |

The two per-model files are checkpoints rather than deliverables, but are worth keeping: they make it possible to rebuild the merged file without re-running a model.

## Findings established here

1. Both transformers score all 498,147 comments, so all four methods in the comparison now cover the same records.
2. RoBERTa: 13.0% positive, 34.9% neutral, 52.1% negative. BERT: 29.3% positive, 8.1% neutral, 62.6% negative.
3. The two transformers agree on 56.2% of comments, barely above the two lexicons' 55.9%.
4. Sample and full-corpus results differ by at most 0.26 percentage points, validating the sampling design used earlier and for the human-coded benchmark.

**Next:** notebook 3 combines all four methods with topic assignments, builds the human-coded benchmark, and selects a model.